# Lab 5 - PySpark

Jakub Kwaśniowski

---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

## 1. Importy i przygotowanie środowiska

In [ ]:
import os
import sys
import glob
import shutil

# PySpark ma używać tego samego Pythona co notebook.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# Szukanie Javy w najczęstszych miejscach.
def znajdz_java_home():
    if os.environ.get("JAVA_HOME"):
        return os.environ["JAVA_HOME"]

    conda_prefix = os.environ.get("CONDA_PREFIX")
    candidates = []

    if conda_prefix:
        candidates.append(os.path.join(conda_prefix, "Library"))

    candidates += glob.glob(r"C:\Program Files\Java\jdk*")
    candidates += glob.glob(r"C:\Program Files\Eclipse Adoptium\jdk*")
    candidates += glob.glob(r"C:\Program Files\Microsoft\jdk*")
    candidates += glob.glob(r"C:\Program Files\OpenJDK\jdk*")

    for path in candidates:
        java_exe = os.path.join(path, "bin", "java.exe")
        if os.path.exists(java_exe):
            return path

    return None


java_home = znajdz_java_home()
if java_home:
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = os.path.join(java_home, "bin") + os.pathsep + os.environ.get("PATH", "")

# Jeżeli winutils.exe jest już w folderze, Spark może go użyć.
# Nie pobieram tutaj żadnych plików .exe z internetu.
if os.name == "nt":
    hadoop_home = os.path.abspath("hadoop_home")
    bin_dir = os.path.join(hadoop_home, "bin")
    winutils_path = os.path.join(bin_dir, "winutils.exe")

    if os.path.exists(winutils_path):
        os.environ["HADOOP_HOME"] = hadoop_home
        os.environ["hadoop.home.dir"] = hadoop_home
        os.environ["PATH"] = bin_dir + os.pathsep + os.environ.get("PATH", "")

print("Python:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME", "brak"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME", "brak"))


## 2. Utworzenie sesji Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("Lab5 Chicago Crimes")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
spark.version


## 3. Wczytanie danych z pliku CSV

In [ ]:
csv_path = "chicago_crimes_sample.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError("Brak pliku chicago_crimes_sample.csv w folderze notebooka")

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(csv_path)
)

# Ujednolicam nazwy kolumn, żeby łatwiej było się do nich odwoływać.
df_raw = df_raw.toDF(*[
    c.strip().lower().replace(" ", "_").replace("/", "_")
    for c in df_raw.columns
])

raw_count = df_raw.count()
print("Liczba rekordów przed czyszczeniem:", raw_count)
df_raw.printSchema()
df_raw.show(5, truncate=False)


## 4. Czyszczenie danych

In [ ]:
df_clean = (
    df_raw
    # Usuwam duplikaty po identyfikatorze zdarzenia.
    .dropDuplicates(["id"])
    # Zamieniam datę na typ timestamp.
    .withColumn(
        "crime_date",
        F.coalesce(
            F.to_timestamp(F.col("date"), "yyyy-MM-dd'T'HH:mm:ss.SSS"),
            F.to_timestamp(F.col("date"), "MM/dd/yyyy hh:mm:ss a"),
            F.to_timestamp(F.col("date"))
        )
    )
    .withColumn("year", F.year(F.col("crime_date")))
)

# Zostawiam tylko rekordy z podstawowymi potrzebnymi danymi.
df_clean = df_clean.dropna(
    subset=["id", "case_number", "crime_date", "primary_type", "location_description", "year"]
)

# Odfiltrowanie błędnych lub podejrzanych lat.
df_clean = df_clean.filter((F.col("year") >= 2001) & (F.col("year") <= 2030))

clean_count = df_clean.count()
print("Liczba rekordów przed czyszczeniem:", raw_count)
print("Liczba rekordów po czyszczeniu:", clean_count)

df_clean.select("id", "case_number", "crime_date", "year", "primary_type", "location_description").show(10, truncate=False)


## 5. UDF - klasyfikacja pory dnia

In [ ]:
def get_day_part(hour):
    if hour is None:
        return "brak"
    if 0 <= hour < 6:
        return "noc"
    if 6 <= hour < 12:
        return "rano"
    if 12 <= hour < 18:
        return "dzien"
    return "wieczor"

get_day_part_udf = F.udf(get_day_part, T.StringType())

df_clean = (
    df_clean
    .withColumn("hour", F.hour("crime_date"))
    .withColumn("pora_dnia", get_day_part_udf(F.col("hour")))
)

df_clean.select("crime_date", "hour", "pora_dnia", "primary_type").show(10, truncate=False)

## 6. Cache danych po czyszczeniu

In [ ]:
# Ten DataFrame będzie używany kilka razy, więc zapisuję go w cache.
df_clean = df_clean.cache()

# count() wymusza materializację cache.
print("Liczba rekordów w cache:", df_clean.count())

## 7. Broadcast join z małą tabelą kategorii

In [ ]:
category_data = [
    ("THEFT", "majatkowe"),
    ("BATTERY", "przeciwko_osobie"),
    ("ASSAULT", "przeciwko_osobie"),
    ("ROBBERY", "majatkowe"),
    ("BURGLARY", "majatkowe"),
    ("MOTOR VEHICLE THEFT", "majatkowe"),
    ("CRIMINAL DAMAGE", "zniszczenie_mienia"),
    ("NARCOTICS", "narkotyki"),
    ("DECEPTIVE PRACTICE", "oszustwa"),
    ("WEAPONS VIOLATION", "bron"),
]

category_df = spark.createDataFrame(category_data, ["primary_type", "crime_category"])

# Tabela kategorii jest mała, dlatego używam broadcast join.
df_final = (
    df_clean
    .join(F.broadcast(category_df), on="primary_type", how="left")
    .fillna({"crime_category": "inne"})
)

df_final.select("primary_type", "crime_category", "location_description", "pora_dnia").show(10, truncate=False)

## 8. Analiza według typu przestępstwa

In [ ]:
crimes_by_type = (
    df_final
    .groupBy("primary_type")
    .count()
    .orderBy(F.desc("count"))
)

crimes_by_type.show(15, truncate=False)

## 9. Analiza według lokalizacji

In [ ]:
crimes_by_location = (
    df_final
    .groupBy("location_description")
    .count()
    .orderBy(F.desc("count"))
)

crimes_by_location.show(15, truncate=False)

## 10. Analiza według pory dnia

In [ ]:
crimes_by_day_part = (
    df_final
    .groupBy("pora_dnia")
    .count()
    .orderBy(F.desc("count"))
)

crimes_by_day_part.show(truncate=False)

## 11. Analiza lokalizacji, pory dnia i typu przestępstwa

In [ ]:
location_time_type = (
    df_final
    .groupBy("location_description", "pora_dnia", "primary_type")
    .count()
    .orderBy(F.desc("count"))
)

location_time_type.show(20, truncate=False)

## 12. Plan wykonania zapytania

In [ ]:
# explain pokazuje plan wykonania najcięższej agregacji.
location_time_type.explain(True)

## 13. Zapis danych do Parquet z partycjonowaniem po roku

In [ ]:
output_path = "chicago_crimes_parquet"

# Usunięcie starego folderu pozwala uruchamiać notebook kilka razy od początku.
if os.path.exists(output_path):
    shutil.rmtree(output_path)

try:
    (
        df_final
        .repartition("year")
        .write
        .mode("overwrite")
        .partitionBy("year")
        .parquet(output_path)
    )
    save_method = "Spark"
except Exception as error:
    print("Zapis przez Spark nie zadziałał w tym środowisku.")
    print("Używam prostego zapisu awaryjnego przez pandas/pyarrow.")
    print("Powód:", str(error).splitlines()[0])

    pdf = df_final.toPandas()
    pdf.to_parquet(
        output_path,
        engine="pyarrow",
        partition_cols=["year"],
        index=False
    )
    save_method = "pandas/pyarrow"

print("Zapisano dane do folderu:", output_path)
print("Metoda zapisu:", save_method)
print("Liczba zapisanych rekordów:", df_final.count())


## 14. Odczyt kontrolny z Parquet

In [ ]:
try:
    df_parquet = spark.read.parquet(output_path)

    print("Liczba rekordów w Parquet:", df_parquet.count())
    df_parquet.groupBy("year").count().orderBy("year").show(truncate=False)
    df_parquet.select("id", "year", "primary_type", "location_description", "pora_dnia").show(10, truncate=False)
except Exception as error:
    print("Odczyt przez Spark nie zadziałał w tym środowisku.")
    print("Używam prostego odczytu kontrolnego przez pandas/pyarrow.")
    print("Powód:", str(error).splitlines()[0])

    import pandas as pd

    pdf_parquet = pd.read_parquet(output_path)
    print("Liczba rekordów w Parquet:", len(pdf_parquet))
    print(pdf_parquet.groupby("year").size().reset_index(name="count"))
    print(pdf_parquet[["id", "year", "primary_type", "location_description", "pora_dnia"]].head(10))


## 15. Zakończenie pracy Sparka

In [ ]:
spark.stop()